In [ ]:
import torch
from torch import optim, nn, pi as π
from torch.nn import functional; nn.f = functional
from utils import set_plot_defaults
from utils.os import setup_save
from ctrl.plot import *
from ctrl.truck_plot import *
from numpy import cos, sin
import matplotlib.pyplot as plt

# Create pth/ and prepend pth/ to save path
save_pth = setup_save(dst='ctrl/pth/')


def truck_dynamics(x, u):
    l = 1           # m how big the cab is
    d = 4           # how big the truck is
    v = -1.0        # m/s constant speed (backing only)

    x, y, theta_cab, theta_truck = x  # m, m, rad, rad
    phi = u[0] if u.ndim else u       # rad steering angle

    f = torch.zeros(4, dtype=x.dtype, device=x.device)

    f[0] = v * torch.cos(theta_cab)      # m/s Cab location update X
    f[1] = v * torch.sin(theta_cab)      # m/s Cab location update Y

    f[2] = v / l * torch.tan(phi)        # rad/s Cab angle update

    f[3] = v / d * torch.sin(theta_cab - theta_truck)  # rad/s Trailer angle update

    return f

def integrate(f, x0, u, dt=1):
    N = len(u)
    x = [0] * (N+1)
    x[0] = x0
    for n in range(N):
        x[n+1] = x[n] + f(x[n], u[n]) * dt
    return torch.stack(x)

# Define LBFGS minimiser, copied from the Ellipse notebook, swapped u for z
def LBFGS(e, u0):
    u0 = nn.Parameter(u0)
    opt = optim.LBFGS((u0,), line_search_fn='strong_wolfe', max_iter=200)
    def closure():
        e_u0 = e(u0)
        opt.zero_grad()
        e_u0.backward()
        return e_u0
    opt.step(closure)

    # Define computation of ǔ
def compute_ǔ(e, N, C=1):
    u = torch.zeros(N, C)  # zero control sequence {u[n]}
    LBFGS(e, u)            # update u as e minimiser
    return u               # return minimiser, i.e. ǔ

# Define cost
C = lambda y, ỹ: (ỹ - y).pow(2).sum()
ER = lambda x, y: torch.norm(x[-1][:2] - y)

def trailer_xy(x):
    d = 4
    x_pos, y_pos, _, theta_truck = x
    xt = x_pos - d * torch.cos(theta_truck)
    yt = y_pos - d * torch.sin(theta_truck)
    return torch.stack((xt, yt))


def cost_truck(y, x):
    N = x.shape[0] - 1    
    w_process_angle = 100.0  
    w_final_pos = 10.0 
    w_final_angle = 200.0 
    
    x_pos, y_pos, theta_cab, theta_truck = x.T

    # state costs
    B = torch.tensor(0.0)
    for n in range(N + 1):
        # Process angle cost
        delta_theta = theta_truck[n] - theta_cab[n]
        delta_theta_limit = 75 * π / 180  # 75 degrees
        violation = torch.relu(torch.abs(delta_theta) - delta_theta_limit)
        B = B + w_process_angle * violation.pow(2)

    # target costs
    # final Position loss
    xt, yt = trailer_xy(x[-1])
    C = w_final_pos * (xt.pow(2) + yt.pow(2)) \
    + w_final_angle * (theta_cab[-1].pow(2) + theta_truck[-1].pow(2))

    return B + C

def error_truck(x, y):
    xt, yt = trailer_xy(x[-1])
    return torch.norm(torch.stack((xt - y[0], yt - y[1], x[-1][3] - y[2])))


def plan_truck(x0, y, N, cost_fn=cost_truck):
    def e(u):
        x = integrate(truck_dynamics, x0, u)
        return cost_fn(y, x)
    ǔ = compute_ǔ(e, N, C=1)
    return ǔ

def plan_line_search(x0, y, N_list=[10, 15, 20, 25, 30, 35, 40, 45, 50], cost_fn=cost_truck):
    best_error = float("inf")
    best_u = None

    for N in N_list:
        u = plan_truck(x0, y, N, cost_fn=cost_fn)
        x = integrate(truck_dynamics, x0, u)
        err = error_truck(x, y)

        print(f"N={N}, terminal error={err.item():.4f}")
        if err < 0.01:
            return u
        
        if err < best_error:
            best_error = err
            best_u = u

    return best_u

# Cost design charts
plot_truck_cost_design(
    w_process_angle=100.0,
    w_final_pos=10.0,
    w_final_angle=200.0,
    delta_theta_limit_deg=75.0,
)


In [ ]:
import torch

# Load dataset
data = torch.load("truck_regression_data.pt")

X_raw = data["X_raw"]              # (num_samples, 4)
N_star = data["N_star"]            # minimal successful horizon or nan
failure_type = data["failure_type"]  # 0=success, 1=terminal, 2=jackknife

y_goal = torch.tensor((0.0, 0.0, 0.0))
eps = 0.5
N_list = [10, 15, 20, 25, 30, 35, 40, 45, 50]

def detect_jackknife(x, limit_deg=90.0):
    theta_c = x[:, 2]
    theta_t = x[:, 3]
    delta_theta = (theta_t - theta_c + torch.pi) % (2*torch.pi) - torch.pi
    limit = limit_deg * torch.pi / 180.0
    return torch.any(torch.abs(delta_theta) > limit)

# Select failing indices
terminal_fail_idx = torch.where(failure_type == 1)[0]
jackknife_fail_idx = torch.where(failure_type == 2)[0]

print("Terminal fails:", len(terminal_fail_idx))
print("Jackknife fails:", len(jackknife_fail_idx))
def replay_term():
    for idx in terminal_fail_idx[:5]:   # limit to first 5
        x0 = X_raw[idx]
        print("\nReplaying terminal failure at index:", idx.item())
        print("Initial state:", x0)

        best_err = float("inf")
        best_x = None

        for N in N_list:
            u = plan_truck(x0, y_goal, N)
            x_traj = integrate(truck_dynamics, x0, u)
            err = error_truck(x_traj, y_goal).item()

            if err < best_err:
                best_err = err
                best_x = x_traj

        print("Best terminal error:", best_err)
        print("Jackknife occurred:", detect_jackknife(best_x).item())
        anim = plot_truck(best_x, y_goal, save_path=f"vis/replay_terminal_{idx}.gif")
        display(anim)

def replay_jack():
    for idx in jackknife_fail_idx[:5]:
        x0 = X_raw[idx]

        print("\nReplaying jackknife failure at index:", idx.item())
        print("Initial state:", x0)

        best_err = float("inf")
        best_x = None
        best_N = None
        best_jack = None

        for N in N_list:
            u = plan_truck(x0, y_goal, N)
            x_traj = integrate(truck_dynamics, x0, u)
            err = error_truck(x_traj, y_goal).item()
            jack = detect_jackknife(x_traj).item()

            if err < best_err:
                best_err = err
                best_x = x_traj
                best_N = N
                best_jack = jack

        print("Best N:", best_N)
        print("Best terminal error:", best_err)
        print("Jackknife occurred:", best_jack)

        anim = plot_truck(
            best_x,
            y_goal,
            save_path=f"vis/replay_jackknife_best_{idx}.gif"
        )
        display(anim)            

replay_term()
replay_jack()